# SHL grammar scoring — final submission report

1. **Training-data evaluation RMSE: 0.477589*** — pooled across **769 labelled recordings and 3 grouped folds**.
2. **Recorded public leaderboard RMSE: 0.3141** — the separate hidden public test evaluation.
3. **Mean of the three fold RMSEs: 0.477121**. Pooled RMSE is the main labelled-data metric because it computes one square root after combining all squared errors.

*This is cross-fitted ensemble-development error over frozen base-model OOF predictions, not an in-sample training-fit score or a fully nested end-to-end validation estimate. Meta-fold preprocessing and fitting excluded each meta-validation partition, but base models were not retrained within every meta-fold. Repeated development search and public-informed final mixture selection can introduce optimism. End-to-end in-sample training RMSE was not recorded. Private-test performance is unknown.*

| Fold (displayed 1–3) | Labelled rows | RMSE |
|---|---:|---:|
| 1 | 256 | 0.447787 |
| 2 | 257 | 0.490571 |
| 3 | 256 | 0.493004 |

This notebook reports the archived submission and verifies its evaluation from private saved prediction arrays. It does not claim a new full GPU retraining run.

## How to read the evaluation

1. **Training-data evaluation: 0.477589 RMSE.** This combines the final ensemble's evaluation errors over 769 labelled recordings and three grouped folds.
2. **Public leaderboard: 0.3141 RMSE, recorded rank 5th.** This score comes from a different set of recordings. The rank is historical, not a live leaderboard check.
3. **OOF means out of fold.** Each base-model OOF prediction was produced while that model held out the recording being predicted. The ensemble used those saved predictions; it did not retrain the whole stack within every outer split.
4. **Development score means the data helped guide model choices.** Repeated searches and public-informed blend selection limit how independently we can interpret the result. An end-to-end in-sample training-fit RMSE was not recorded.


## Research methodology

1. I began by reviewing related Kaggle competitions involving grammar and language-quality scoring to understand successful approaches and their limitations. I then studied arXiv research on audio processing and transformer representations, looking for ideas applicable to spoken grammar assessment. These findings guided a documented experiment plan: establish a simple baseline, compare transcription methods, explore text and audio representations, and combine complementary models. I used grouped cross-validation, RMSE, and error analysis to assess each iteration, while distinguishing improvements in public leaderboard performance from evidence of broader generalization.
2. Related competition work informed the text/pooling experiments. Audio-representation research informed the use of intermediate and final speech-transformer layers. Neither establishes an improvement without a task-specific comparison.
3. I recorded hypotheses and experiment outcomes, including unsuccessful directions, rather than treating every researched technique as a completed improvement. The [research journey](docs/RESEARCH.md), [references](docs/REFERENCES.md), and [what worked / what did not](docs/WHAT_WORKED.md) connect the methodology to the evidence.

## 1. Approach, preprocessing, and architecture

1. Start with Whisper transcripts and a word/character TF-IDF + Ridge baseline. Preserve grammatical errors, repetitions, and disfluencies rather than grammar-correcting the input.
2. Combine DeBERTa contextual text representations with selected WavLM/HuBERT/wav2vec2 layer summaries and full-recording log-mel spectrograms. Join inputs by both split and filename; fit imputation and scaling on training folds.
3. The audited AST fusion model uses DeBERTa-v3-large masked mean pooling, 11 speech channels totalling 8,448 features, a 10-layer gated MLP of width 2,048, and an AudioSet-pretrained AST spectrogram branch. Concatenation and a bounded regression head predict scores in [0,5]. Train with MSE, select epochs with validation RMSE.
4. Use grouped validation to keep known related source recordings together. Later source discoveries exposed residual grouping limitations; no scores are silently reinterpreted under new folds.
5. The archived final ensemble is **60% learned 14-input hurdle + 40% retained 12-input joint-quality ensemble**. The learned hurdle predicts zero/nonzero probability from base predictions and combines it with positive-score residual regression. It is a separate meta-model, not a zero head in the neural architecture.
6. The hurdle achieved the lowest recorded public RMSE but may exploit the unusual zero-labelled subset. It is retained as an experimental benchmark result rather than asserted to be the best-generalizing approach. The earlier non-hurdle quadratic/kernel submission scored 0.3166 publicly.

![Architecture](docs/assets/architecture.svg)

Full descriptions: [architecture](docs/ARCHITECTURE.md), [ensemble mechanics](docs/ENSEMBLES.md), [generalization](docs/GENERALIZATION.md), [what worked and did not](docs/WHAT_WORKED.md).

## 2. Settings and private inputs

1. Run this notebook from the repository root, or set `SHL_REPOSITORY` to its location. Set `SHL_PRIVATE_ARTIFACTS` to your authorized saved artifacts directory.
2. The public repository contains aggregate metrics and code. Original labels, per-record predictions, and competition data remain private.
3. The verification below needs the original `inputs.npz`, hurdle development/test predictions, and retained joint development/test predictions, in their recorded directory layout. It checks all shapes, finite bounded values, fold membership and aggregate scores.
4. If private arrays are absent, the notebook explicitly displays the archived report; it does not pretend to recompute the score.

In [ ]:
from pathlib import Path
import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(os.environ.get("SHL_REPOSITORY", str(Path.cwd()))).resolve()
PRIVATE = Path(os.environ.get("SHL_PRIVATE_ARTIFACTS", str(ROOT.parent / "artifacts"))).resolve()
OUTPUT = ROOT / "outputs" / "final_submission"
report = json.loads((ROOT / "reports/final_training_data_evaluation.json").read_text())
paths = {
    "inputs": PRIVATE / "rmse_research_20261015/inputs.npz",
    "hurdle_dev": PRIVATE / "rmse_research_20261015/hurdle_development_predictions.npz",
    "joint_dev": PRIVATE / "all_metadata_ensembles_20261015/development_predictions.npz",
    "hurdle_test": PRIVATE / "rmse_research_20261015/hurdle_test_predictions.npz",
    "joint_test": PRIVATE / "all_metadata_ensembles_20261015/full_predictions.npz",
}
can_verify = all(path.is_file() for path in paths.values())
print("Private-array verification available:", can_verify)
if not can_verify:
    print("Archived aggregate report only; set SHL_PRIVATE_ARTIFACTS to recompute.")
display(pd.DataFrame(report["folds"]).assign(fold=lambda x: x.fold + 1))
print(f"Recorded training-data pooled development RMSE: {report['pooled_rmse']:.9f}")
print(f"Recorded mean fold RMSE: {report['mean_fold_rmse']:.9f}")
print(f"Recorded public leaderboard RMSE: {report['public_rmse']:.4f}")

## 3. Recompute training-data RMSE

1. For each labelled row, use the fixed 60/40 combination of its archived cross-fitted meta-model predictions.
2. Compute `sqrt(mean((true_score - prediction)**2))` over all 769 rows. This is pooled RMSE, not the arithmetic average of the fold RMSEs.
3. This recomputes the historical development metric; it does not remove selection optimism or make frozen-base stacking fully nested.

In [ ]:
if can_verify:
    # Never use test labels. Targets below belong only to the labelled training set.
    with np.load(paths["inputs"], allow_pickle=False) as data:
        y = data["y"].copy()
        folds = data["folds"].copy()
        groups = data["groups"].copy()
        test_names = data["test_names"].copy()
    with np.load(paths["hurdle_dev"], allow_pickle=False) as data:
        hurdle = data["hurdle_regression"].copy()
    with np.load(paths["joint_dev"], allow_pickle=False) as data:
        joint = data["joint_quality"].copy()
    assert y.shape == hurdle.shape == joint.shape == folds.shape == (769,)
    assert len(np.unique(folds)) == 3
    assert all(len(np.unique(folds[groups == group])) == 1 for group in np.unique(groups))
    prediction = 0.60 * hurdle + 0.40 * joint
    assert np.isfinite(prediction).all() and ((prediction >= 0) & (prediction <= 5)).all()
    pooled = float(np.sqrt(np.mean((y - prediction) ** 2)))
    fold_scores = [{"fold": int(fold) + 1, "rows": int((folds == fold).sum()),
                    "rmse": float(np.sqrt(np.mean((y[folds == fold] - prediction[folds == fold]) ** 2)))}
                   for fold in sorted(np.unique(folds))]
    mean_fold = float(np.mean([row["rmse"] for row in fold_scores]))
    np.testing.assert_allclose(pooled, report["pooled_rmse"], atol=1e-12, rtol=0)
    np.testing.assert_allclose(mean_fold, report["mean_fold_rmse"], atol=1e-12, rtol=0)
    display(pd.DataFrame(fold_scores))
    print(f"VERIFIED training-data pooled development RMSE: {pooled:.9f}")
    print(f"VERIFIED mean fold RMSE: {mean_fold:.9f}")
else:
    print("Not recomputed: private archived arrays are unavailable.")

## 4. Reconstruct the submitted output

1. Apply the same 60/40 combination to the two archived test-prediction arrays. Check the actual 216 test IDs and required `filename,label` columns.
2. Where the original archived CSV is present, check its hash and numerical agreement, then copy its exact bytes into `prabal_singh.csv`. Otherwise write the reconstructed predictions and label the output accordingly.
3. To refit the saved meta-model parameters from their ordered base-prediction inputs instead of replaying cached component outputs, use [scripts/predict_final.py](scripts/predict_final.py). Full base training is documented separately in [the reproduction guide](docs/REPRODUCING.md).

In [ ]:
if can_verify:
    with np.load(paths["hurdle_test"], allow_pickle=False) as data:
        hurdle_test = data["hurdle_regression"].copy()
    with np.load(paths["joint_test"], allow_pickle=False) as data:
        joint_test = data["joint_quality"].copy()
    test_prediction = 0.60 * hurdle_test + 0.40 * joint_test
    assert test_prediction.shape == test_names.shape == (216,)
    assert len(set(test_names.tolist())) == 216
    assert np.isfinite(test_prediction).all()
    assert ((test_prediction >= 0) & (test_prediction <= 5)).all()
    frame = pd.DataFrame({"filename": test_names, "label": test_prediction})
    manifest = json.loads((ROOT / "reports/winner_manifest.json").read_text())
    original = PRIVATE / "public_best_20261015_03141" / manifest["submission"]
    OUTPUT.mkdir(parents=True, exist_ok=True)
    destination = OUTPUT / "prabal_singh.csv"
    if original.is_file():
        assert hashlib.sha256(original.read_bytes()).hexdigest() == manifest["sha256"]
        reference = pd.read_csv(original)
        assert reference.filename.tolist() == frame.filename.tolist()
        np.testing.assert_allclose(reference.label, test_prediction, atol=1e-12, rtol=0)
        shutil.copyfile(original, destination)
        print("Exported exact archived submission bytes.")
    else:
        frame.to_csv(destination, index=False)
        print("Exported reconstructed predictions; original CSV byte identity not checked.")
    print("Output:", destination.name)
    print("Rows:", len(frame))
    print("SHA256:", hashlib.sha256(destination.read_bytes()).hexdigest())
else:
    print("No output generated: private test predictions are unavailable.")

## 5. Results, hardware, limitations, and references

1. The labelled-data development RMSE is **0.477589** over **3 grouped folds**, while the recorded public RMSE is **0.3141**. They evaluate different populations and should not be subtracted as a direct train/test gap. An end-to-end in-sample training-fit RMSE is not available in the saved record.
2. The audited AST run used RTX 5090 on Vast.ai; most later base runs used RTX 4090. The original transcription workflow used Kaggle T4 ×2. The audited neural settings include BF16, gradient checkpointing, microbatch 8, and accumulation 2. See [hardware records](reports/training_hardware.json).
3. Source grouping, fold-local preprocessing, early stopping, and regularized meta-model fits reduce specific overfitting risks. They do not eliminate unknown speaker overlap, later-discovered source relationships, repeated search, or public-informed selection.
4. The hurdle result is sensitive to the unusual zero subset and does not establish deployment generalization. A frozen recipe evaluated on unseen sources/speakers is the stronger next test.
5. ![Local versus public evaluation](docs/assets/score_comparison.svg)
6. ![Training curves](docs/assets/learning_curves.svg)
7. [Research references](docs/REFERENCES.md) explain intermediate-layer speech representations, DeBERTa/pooling inspiration, AST, speech backbones, and grouped stacking. The [README](README.md) and [reproduction guide](docs/REPRODUCING.md) provide the complete project account.
8. AI tools assisted implementation and analysis. Metrics are reconstructed from the saved evidence; no missing training score or new independent validation result is invented.